# FPGA board test of the 4×4 systolic array (`host_test.py`)

This notebook is the **PC side** of the system. It sends signed INT8 matrices A and B to the Basys 3
over the USB-UART, reads back C = A × B (16 signed 32-bit values, most significant byte first) and
checks it against NumPy.

**Before running:**

1. Program the Basys 3 with the bitstream of the 4×4 design (Vivado → *Open Hardware Manager* →
   *Program Device*).
2. Close any other program using the port (a serial terminal, or the Vivado hardware manager if it
   holds the UART).
3. Set `PORT` in Section 1. On Windows look in *Device Manager → Ports (COM & LPT) → USB Serial Port
   (COMx)*; on Linux it is usually `/dev/ttyUSB1`.

Without a programmed board, the connection cell (or the first job) fails with a serial error or a
timeout. That means nothing answered, not that the design is wrong.

## 0. Install the packages (once)

In [ ]:
%pip install pyserial numpy

## 1. Settings

In [ ]:
import time
import numpy as np
import serial

PORT = "COM5"          # change to your board's port, e.g. "COM3" or "/dev/ttyUSB1"
BAUD = 115200          # must match the design (CLKS_PER_BIT = 868 at 100 MHz)
N = 4                  # 8 for the 8x8 design
NUM_RANDOM = 200       # number of random tests in Section 5
SEED = 1               # change for a different set of random matrices

N_IN, N_OUT = 2 * N * N, 4 * N * N     # 32 bytes in, 64 bytes out for 4x4
print(f"{N}x{N}: send {N_IN} bytes, expect {N_OUT} bytes per job")

## 2. Open the serial port

`timeout=1.0` means a read gives up after one second. A job needs only about 8.3 ms (4×4) or
33.3 ms (8×8) of UART time, so one second is plenty.

In [ ]:
ser = serial.Serial(PORT, BAUD, timeout=1.0)
time.sleep(0.1)
print("opened", ser.name)

## 3. One job

* `tobytes()` turns each matrix into its INT8 bytes, row by row: A first, then B (2N² bytes).
* `reset_input_buffer()` throws away any stray bytes left from earlier.
* `read(N_OUT)` waits for the 4N² result bytes.
* `dtype=">i4"` reads them as **big-endian** (`>`) signed 32-bit integers (`i4`), the byte order the
  FPGA sends.

In [ ]:
def run_job(ser, A, B):
    """Send one (A, B) pair, return C read back from the FPGA."""
    payload = A.astype(np.int8).tobytes() + B.astype(np.int8).tobytes()
    ser.reset_input_buffer()
    ser.write(payload)
    raw = ser.read(N_OUT)
    if len(raw) != N_OUT:
        raise TimeoutError(f"expected {N_OUT} bytes, got {len(raw)} - check the port / reset the board")
    return np.frombuffer(raw, dtype=">i4").reshape(N, N)

## 4. Demo: one readable example

Small numbers (−10 to 9) so the result is easy to read. The LEDs on the board should flash
(LED1 compute, LED2 transmit) and the job counter on LEDs 15–8 goes up by one.

In [ ]:
rng = np.random.default_rng(SEED)
A = rng.integers(-10, 10, (N, N))
B = rng.integers(-10, 10, (N, N))
C = run_job(ser, A, B)
print("A =\n", A, "\n\nB =\n", B)
print("\nFPGA C =\n", C, "\n\nNumPy A@B =\n", A @ B)
print("\nMATCH" if np.array_equal(C, A @ B) else "\nMISMATCH")

## 5. Full test: corner cases + random matrices

Corner cases:

* all −128 × all −128: largest positive result (N × 16 384 = 65 536 for 4×4),
* all +127 × all −128: largest negative result,
* zero matrix × random matrix: result must be all zeros,
* identity × a known matrix: result must equal that matrix.

Then `NUM_RANDOM` random INT8 pairs. The reference uses 64-bit integers, so NumPy itself cannot
overflow.

In [ ]:
def corner_cases():
    yield np.full((N, N), -128), np.full((N, N), -128)
    yield np.full((N, N), 127),  np.full((N, N), -128)
    yield np.zeros((N, N), dtype=int), np.random.randint(-128, 128, (N, N))
    yield np.eye(N, dtype=int), np.arange(N * N).reshape(N, N) - N * N // 2

rng = np.random.default_rng(SEED)
tests = list(corner_cases())
tests += [(rng.integers(-128, 128, (N, N)), rng.integers(-128, 128, (N, N))) for _ in range(NUM_RANDOM)]

fails = 0
t0 = time.time()
for idx, (A, B) in enumerate(tests):
    C_fpga = run_job(ser, A, B)
    C_ref = A.astype(np.int64) @ B.astype(np.int64)
    if not np.array_equal(C_fpga, C_ref):
        fails += 1
        print(f"[FAIL] test {idx}\nA=\n{A}\nB=\n{B}\nfpga=\n{C_fpga}\nref=\n{C_ref}")
dt = time.time() - t0

print(f"{len(tests) - fails}/{len(tests)} tests passed in {dt:.1f} s "
      f"({len(tests) / dt:.1f} jobs/s, limited by the UART)")

**Expected:** `204/204 tests passed`. The jobs per second will be roughly 1 / 8.3 ms ≈ 120 for 4×4
(about 30 for 8×8) minus USB overhead, which shows that the UART, not the array, limits the speed.

## 6. Close the port

Always close it, or the port stays busy until the kernel is restarted.

In [ ]:
ser.close()
print("port closed")